In [3]:
from pynput import mouse
import time
import json

# LOAD FILE
with open("mouse_actions.json", "r") as f:
    actions = json.load(f)

controller = mouse.Controller()

print("Replaying in 3 seconds...")
time.sleep(3)

for i, action in enumerate(actions):
    if i > 0:
        delay = action[-1] - actions[i-1][-1]
        time.sleep(delay)

    if action[0] == "move":
        _, x, y, _ = action
        controller.position = (x, y)

    elif action[0] == "click":
        _, x, y, button, pressed, _ = action
        controller.position = (x, y)

        # Convert string back to button
        if "left" in button:
            btn = mouse.Button.left
        elif "right" in button:
            btn = mouse.Button.right
        else:
            btn = mouse.Button.middle

        if pressed:
            controller.press(btn)
        else:
            controller.release(btn)

    elif action[0] == "scroll":
        _, x, y, dx, dy, _ = action
        controller.scroll(dx, dy)

print("Replay finished.")

Replaying in 3 seconds...
Replay finished.


In [1]:
from pynput import mouse
import time
import json

actions = []
recording = True

def on_move(x, y):
    if recording:
        actions.append(("move", x, y, time.time()))

def on_click(x, y, button, pressed):
    if recording:
        actions.append(("click", x, y, str(button), pressed, time.time()))
    if button == mouse.Button.right and not pressed:
        return False

def on_scroll(x, y, dx, dy):
    if recording:
        actions.append(("scroll", x, y, dx, dy, time.time()))

print("Recording... Right-click to stop.")
start_time = time.time()

with mouse.Listener(on_move=on_move, on_click=on_click, on_scroll=on_scroll) as listener:
    listener.join()

# Normalize time
normalized_actions = []
for action in actions:
    new_action = list(action)
    new_action[-1] -= start_time
    normalized_actions.append(new_action)

# SAVE TO FILE
with open("mouse_actions.json", "w") as f:
    json.dump(normalized_actions, f, indent=4)

print("Saved to mouse_actions.json")

Recording... Right-click to stop.
Saved to mouse_actions.json


In [ ]:
import json
import time
from pynput import mouse, keyboard

events = []
start_time = time.time()
stop_recording = False

# Track pressed keys for combination detection
pressed_keys = set()

def get_time():
    return round(time.time() - start_time, 4)

# ---------------- MOUSE ----------------
def on_move(x, y):
    if stop_recording:
        return False
    events.append({
        "type": "mouse_move",
        "time": get_time(),
        "x": x,
        "y": y
    })

def on_click(x, y, button, pressed):
    if stop_recording:
        return False
    events.append({
        "type": "mouse_click",
        "time": get_time(),
        "x": x,
        "y": y,
        "button": str(button),
        "pressed": pressed
    })

def on_scroll(x, y, dx, dy):
    if stop_recording:
        return False
    events.append({
        "type": "mouse_scroll",
        "time": get_time(),
        "x": x,
        "y": y,
        "dx": dx,
        "dy": dy
    })

# ---------------- KEYBOARD ----------------
def on_press(key):
    global stop_recording

    pressed_keys.add(key)

    # Detect SHIFT + 2
    if (keyboard.Key.shift in pressed_keys or keyboard.Key.shift_l in pressed_keys or keyboard.Key.shift_r in pressed_keys):
        try:
            if key.char == '2':
                print("Stopping recording...")
                stop_recording = True
                return False
        except AttributeError:
            pass

    events.append({
        "type": "key_press",
        "time": get_time(),
        "key": str(key)
    })

def on_release(key):
    if key in pressed_keys:
        pressed_keys.remove(key)

    events.append({
        "type": "key_release",
        "time": get_time(),
        "key": str(key)
    })

# ---------------- START LISTENERS ----------------
mouse_listener = mouse.Listener(
    on_move=on_move,
    on_click=on_click,
    on_scroll=on_scroll
)

keyboard_listener = keyboard.Listener(
    on_press=on_press,
    on_release=on_release
)

print("Recording started... Press SHIFT + 2 to stop.")

mouse_listener.start()
keyboard_listener.start()

keyboard_listener.join()  # Wait until stop key is pressed
mouse_listener.stop()

# ---------------- SAVE TO JSON ----------------
with open("recorded_events.json", "w") as f:
    json.dump(events, f, indent=4)

print("Recording saved to recorded_events.json")

Recording started... Press SHIFT + 2 to stop.
